**Lab type:** write  
**Course:** DS105 — Exploratory Data Analysis  
**Lesson:** Finding and Handling Outliers  
**Task:** Complete each stub to build a full outlier-detection and handling pipeline. After each section, answer the interpretation question in the comment cell below it.

## Setup: Load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(7)
n = 83600

channel = np.random.choice(
    ['web', 'mobile', 'in-store', 'phone'],
    size=n,
    p=[0.45, 0.35, 0.15, 0.05]
)

status = np.random.choice(
    ['delivered', 'pending', 'shipped', 'cancelled', 'processing'],
    size=n,
    p=[0.728, 0.129, 0.097, 0.035, 0.011]
)

# quantity: mostly 1–5 units, with 18 injected bulk orders (50–500 units)
quantity = np.random.choice([1, 2, 3, 4, 5], n, p=[0.4, 0.3, 0.15, 0.1, 0.05])
bulk_idx = np.random.choice(n, 18, replace=False)
quantity[bulk_idx] = np.random.randint(50, 501, 18)

unit_price = np.round(
    np.random.choice([18.99, 29.99, 39.99, 49.99, 69.99, 99.99], n), 2
)

# revenue: right-skewed; cancelled orders have revenue = 0
revenue = np.round(quantity * unit_price * np.random.uniform(0.85, 1.0, n), 2)
revenue[status == 'cancelled'] = 0.0

# Inject 12 sentinel rows: unit_price = 9999 (data entry error)
sentinel_idx = np.random.choice(np.where(status != 'cancelled')[0], 12, replace=False)
unit_price[sentinel_idx] = 9999.0
revenue[sentinel_idx] = quantity[sentinel_idx] * 9999.0

order_date = pd.date_range(start='2023-01-01', periods=n, freq='1h')[:n]

df = pd.DataFrame({
    'order_id':    ['ORD-{:06d}'.format(i) for i in range(1, n + 1)],
    'order_date':  order_date,
    'quantity':    quantity,
    'unit_price':  unit_price,
    'revenue':     revenue,
    'channel':     channel,
    'status':      status,
})

print(f'Shape: {df.shape}')
print(df[['quantity', 'unit_price', 'revenue']].describe().round(2))

---

## Step 1: Detect outliers with the IQR method

Write a function `flag_outliers(series)` that returns the subset of values that fall below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`. Apply it to `revenue` and `quantity`, and print the count and percentage of flagged rows for each.

In [ ]:
def flag_outliers(series):
    # YOUR CODE HERE
    pass

# Apply to revenue
# YOUR CODE HERE

# Apply to quantity
# YOUR CODE HERE

**Interpretation:** Do the outlier counts look plausible given the dataset description above? Are the percentages similar for `revenue` and `quantity`, or very different — and why might they differ?

---

## Step 2: Detect outliers with the z-score method

Use `scipy.stats.zscore` to find rows in `revenue` where the absolute z-score exceeds 3. Print the count of flagged rows.

Then compare the count to what the IQR method found in Step 1. Which method flags more rows?

In [ ]:
# Compute absolute z-scores for revenue (drop NaN before computing)
# YOUR CODE HERE

# Print number of z-score outliers
# YOUR CODE HERE

# Compare: which method finds more outliers in revenue?
# YOUR CODE HERE

**Interpretation:** `revenue` is right-skewed. How does skew affect z-score detection? Which method is more appropriate here, and why?

---

## Step 3: Visualise outliers with box plots

Create a figure with two side-by-side box plots of `revenue`:
- Left: raw values
- Right: log-transformed values using `np.log1p`

Use `seaborn.boxplot` and add a title to each axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left box plot: raw revenue
# YOUR CODE HERE

# Right box plot: log-transformed revenue
# YOUR CODE HERE

plt.tight_layout()
plt.show()

**Interpretation:** What does the log-transformed plot reveal that the raw plot obscures? Are the flagged points isolated dots far from the rest, or do they sit at the end of a continuous tail?

---

## Step 4: Investigate — the three questions

Before deciding what to do with the revenue outliers, answer the three questions from the lesson.

### Question 1: Are they real values?

Print the top 10 highest-revenue rows, showing `order_id`, `quantity`, `unit_price`, `channel`, and `revenue`.

In [ ]:
# Print the 10 highest-revenue rows with the columns listed above
# YOUR CODE HERE

**Answer:** Based on the output, are the high-revenue rows plausible? Is there any evidence of a data entry error or sentinel value?

### Question 2: Do they follow a pattern?

For rows where `revenue > 5000`:
1. Print the `channel` value counts
2. Print the count by month (use `order_date` converted to period `'M'`)

In [ ]:
high_rev = df[df['revenue'] > 5000]

# Channel distribution of high-revenue rows
# YOUR CODE HERE

# Monthly distribution of high-revenue rows
# YOUR CODE HERE

**Answer:** Are the high-revenue rows evenly spread across channels and time, or concentrated? What does that tell you about whether they represent a real sub-population?

---

## Step 5: Handle outliers

Based on your investigation, apply two separate treatments:

1. **Cap at the 99th percentile:** Create a column `revenue_capped` using `.clip(upper=...)`. Print the original max and the capped max.
2. **Remove sentinel rows:** Filter out any row where `unit_price == 9999`. Store the result in `df_clean` and print the number of rows removed.

In [ ]:
# 1. Cap revenue at the 99th percentile
# YOUR CODE HERE

print(f"Original max: {df['revenue'].max():.0f}")
# YOUR CODE HERE — print capped max

In [ ]:
# 2. Remove sentinel rows (unit_price == 9999)
# YOUR CODE HERE

# Print how many rows were removed
# YOUR CODE HERE

**Interpretation:** Why are capping and removal the right choices for these two groups? What would go wrong if you removed all IQR-flagged rows instead?

---

## Step 6: Spot multivariate outliers

Create a scatter plot of `quantity` (x-axis) vs `revenue` (y-axis) using `seaborn.scatterplot` with `alpha=0.3`. Use `df_clean` from Step 5 so the sentinel rows are already removed.

Look for points that sit far from the main cluster.

In [ ]:
# Scatter plot: quantity vs revenue
# YOUR CODE HERE

plt.title('Revenue vs Quantity')
plt.show()

**Interpretation:** Do the extreme-quantity rows (bulk orders) sit on or near the same revenue-per-unit trend as the rest of the data? If so, what does that suggest about their legitimacy?